<a href="https://colab.research.google.com/github/luvbenz/2025-CV/blob/hw3/hw3_cv_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 기본 라이브러리 임포트
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms   # transforms 포함
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# 데이터 전처리 정의 (Tensor 변환 및 정규화)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # 평균 0.5, 표준편차 0.5
])

In [ ]:
# 데이터셋 로딩
train_data = datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_data = datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

In [ ]:
# DataLoader 생성
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

In [ ]:
# 이미지 시각화(테스트)
examples = enumerate(train_loader)
batch_idx, (example_data, example_targets) = next(examples)

plt.imshow(example_data[0][0], cmap='gray')
plt.title(f"Label: {example_targets[0].item()}")
plt.show()

In [ ]:
# MLP 모델 정의
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = x.view(-1, 784)  # 28x28 이미지를 784 벡터로 펼침
        return self.model(x)

In [ ]:
# 학습, 평가 함수 정의
def train(model, loader, loss_fn, optimizer, use_softmax=False):
    model.train()
    total_loss = 0
    correct = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        output = model(x)

        if use_softmax:
            output = torch.softmax(output, dim=1)
            y_onehot = F.one_hot(y, num_classes=10).float()
            loss = loss_fn(output, y_onehot)
        else:
            loss = loss_fn(output, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = output.argmax(1)
        correct += pred.eq(y).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc


def evaluate(model, loader, loss_fn, use_softmax=False):
    model.eval()
    total_loss = 0
    correct = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            output = model(x)

            if use_softmax:
                output = torch.softmax(output, dim=1)
                y_onehot = F.one_hot(y, num_classes=10).float()
                loss = loss_fn(output, y_onehot)
            else:
                loss = loss_fn(output, y)

            total_loss += loss.item()
            pred = output.argmax(1)
            correct += pred.eq(y).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

In [ ]:
# 실험 A 실행(CrossEntropy vs MSE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 20

# 결과 저장용
results_ce = []
results_mse = []

# CrossEntropyLoss 실험
model_ce = MLP().to(device)
optimizer_ce = optim.SGD(model_ce.parameters(), lr=0.1)
loss_fn_ce = nn.CrossEntropyLoss()

for epoch in range(epochs):
    train_loss, train_acc = train(model_ce, train_loader, loss_fn_ce, optimizer_ce)
    test_loss, test_acc = evaluate(model_ce, test_loader, loss_fn_ce)
    results_ce.append((train_acc, test_acc))
    print(f"[CE] Epoch {epoch+1} - TrainAcc: {train_acc:.4f}, TestAcc: {test_acc:.4f}")


# MSELoss + Softmax 실험
model_mse = MLP().to(device)
optimizer_mse = optim.SGD(model_mse.parameters(), lr=0.1)
loss_fn_mse = nn.MSELoss()

for epoch in range(epochs):
    train_loss, train_acc = train(model_mse, train_loader, loss_fn_mse, optimizer_mse, use_softmax=True)
    test_loss, test_acc = evaluate(model_mse, test_loader, loss_fn_mse, use_softmax=True)
    results_mse.append((train_acc, test_acc))
    print(f"[MSE] Epoch {epoch+1} - TrainAcc: {train_acc:.4f}, TestAcc: {test_acc:.4f}")


In [ ]:
# Accuracy 결과 추출
train_ce = [x[0] for x in results_ce]
test_ce = [x[1] for x in results_ce]
train_mse = [x[0] for x in results_mse]
test_mse = [x[1] for x in results_mse]

In [ ]:
from tabulate import tabulate

headers = ["Loss Type", "Final Train Acc", "Final Test Acc"]
table = [
    ["CrossEntropy", train_ce[-1], test_ce[-1]],
    ["MSE + Softmax", train_mse[-1], test_mse[-1]]
]

print(tabulate(table, headers=headers, floatfmt=".4f"))

In [ ]:
# Accuracy 시각화
plt.figure(figsize=(10, 5))
plt.plot(train_ce, label="Train Accuracy - CrossEntropy")
plt.plot(test_ce, label="Test Accuracy - CrossEntropy")
plt.plot(train_mse, label="Train Accuracy - MSE+Softmax")
plt.plot(test_mse, label="Test Accuracy - MSE+Softmax")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Comparison: CrossEntropy vs MSE")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 데이터 생성 및 정규화
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 텐서 변환
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

# train/test 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# DataLoader로 변환
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1000)

In [ ]:
# MLP 모델 정의
class MLP_Act(nn.Module):
    def __init__(self, activation_fn):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 16),
            activation_fn(),
            nn.Linear(16, 8),
            activation_fn(),
            nn.Linear(8, 2)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# 학습, 평가함수 정의
def train(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, correct = 0, 0
    for x, y in loader:
        optimizer.zero_grad()
        output = model(x)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pred = output.argmax(1)
        correct += pred.eq(y).sum().item()
    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            loss = loss_fn(output, y)
            total_loss += loss.item()
            pred = output.argmax(1)
            correct += pred.eq(y).sum().item()
    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

In [ ]:
# 실험 B 실행 (ReLU / LeakyReLU / Sigmoid)
activations = {
    "ReLU": nn.ReLU,
    "LeakyReLU": nn.LeakyReLU,
    "Sigmoid": nn.Sigmoid
}

epochs = 100
results = {}

loss_fn = nn.CrossEntropyLoss()

for name, act_fn in activations.items():
    model = MLP_Act(act_fn)
    optimizer = optim.SGD(model.parameters(), lr=0.1)
    train_accs = []
    test_accs = []
    for epoch in range(epochs):
        train_loss, train_acc = train(model, train_loader, loss_fn, optimizer)
        test_loss, test_acc = evaluate(model, test_loader, loss_fn)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
    results[name] = (train_accs, test_accs)
    print(f"{name} 최종 Test Acc: {test_accs[-1]:.4f}")


In [ ]:
# 결과 시각화
plt.figure(figsize=(10, 5))
for name in results:
    plt.plot(results[name][1], label=f"{name} Test Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Activation Function Comparison")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Dead ReLU 비율 측정
def count_dead_relu(model, data_loader):
    model.eval()
    with torch.no_grad():
        for x, _ in data_loader:
            x = model.model[0](x)  # 첫 Linear
            a1 = model.model[1](x)  # 활성화
            dead = (a1 == 0).float().mean().item()
            return dead  # 첫 레이어에 대해서만 계산

for name, act_fn in activations.items():
    if name == "ReLU":
        model = MLP_Act(act_fn)
        optimizer = optim.SGD(model.parameters(), lr=0.1)
        for epoch in range(epochs):
            train(model, train_loader, loss_fn, optimizer)
        dead_ratio = count_dead_relu(model, train_loader)
        print(f"{name} Dead ReLU 비율: {dead_ratio:.4f}")

In [ ]:
# Fashion-MNIST 데이터셋 로딩(Fashion-MNIST 기준)
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 데이터 전처리
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 데이터셋 로딩
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

In [ ]:
# MLP 모델 정의
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)  # (N, 1, 28, 28) → (N, 784)
        return self.model(x)

In [ ]:
# 학습 및 평가 함수 정의
def train(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        output = model(x)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(y).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = loss_fn(output, y)

            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(y).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

In [ ]:
# Optimizer 비교 실험 루프
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn = nn.CrossEntropyLoss()
epochs = 20

optimizers = {
    "SGD": lambda model: optim.SGD(model.parameters(), lr=0.1),
    "SGD+Momentum": lambda model: optim.SGD(model.parameters(), lr=0.05, momentum=0.9),
    "Adam": lambda model: optim.Adam(model.parameters(), lr=0.001)
}

results = {}

for name, opt_fn in optimizers.items():
    print(f"▶ Optimizer: {name}")
    model = MLP().to(device)
    optimizer = opt_fn(model)

    train_accs, test_accs, losses = [], [], []

    for epoch in range(epochs):
        train_loss, train_acc = train(model, train_loader, loss_fn, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)

        train_accs.append(train_acc)
        test_accs.append(test_acc)
        losses.append(test_loss)

        print(f"[{name}] Epoch {epoch+1}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}, Loss={test_loss:.4f}")

    results[name] = {
        "train_acc": train_accs,
        "test_acc": test_accs,
        "test_loss": losses
    }


In [ ]:
# Optimizer별 정확도 및 손실 그래프 시각화
# 정확도 비교
plt.figure(figsize=(10, 5))
for name in results:
    plt.plot(results[name]["test_acc"], label=f"{name} Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Optimizer Comparison - Test Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# 손실 비교
plt.figure(figsize=(10, 5))
for name in results:
    plt.plot(results[name]["test_loss"], label=f"{name} Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Optimizer Comparison - Test Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Adam + 지수 감소(ExponentialLR) 실험
# 새 모델 및 옵티마이저 정의
model_decay = MLP().to(device)
optimizer_decay = optim.Adam(model_decay.parameters(), lr=0.01)

# ExponentialLR 적용 (감쇠율 gamma=0.9)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer_decay, gamma=0.9)

# 학습 기록 리스트
decay_acc = []
decay_loss = []

# 학습 루프
for epoch in range(epochs):
    train_loss, train_acc = train(model_decay, train_loader, loss_fn, optimizer_decay, device)
    test_loss, test_acc = evaluate(model_decay, test_loader, loss_fn, device)

    decay_acc.append(test_acc)
    decay_loss.append(test_loss)

    scheduler.step()  # 학습률 감쇠 적용
    print(f"[Adam + LR Decay] Epoch {epoch+1}: Test Acc={test_acc:.4f}, Loss={test_loss:.4f}")

In [ ]:
# 시각화 - Adam vs Adam + Decay 비교
# 정확도 비교
plt.figure(figsize=(10, 5))
plt.plot(results["Adam"]["test_acc"], label="Adam")
plt.plot(decay_acc, label="Adam + LR Decay", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Adam vs Adam + LR Decay - Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# 손실 비교
plt.figure(figsize=(10, 5))
plt.plot(results["Adam"]["test_loss"], label="Adam")
plt.plot(decay_loss, label="Adam + LR Decay", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Adam vs Adam + LR Decay - Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 정량적 분석
from tabulate import tabulate

summary_table = []

# 기본 Optimizer 3종
for name in results:
    acc_list = results[name]["test_acc"]
    loss_list = results[name]["test_loss"]
    final_acc = acc_list[-1]
    min_loss = min(loss_list)
    best_epoch = acc_list.index(max(acc_list)) + 1  # 1부터 시작
    summary_table.append([name, final_acc, min_loss, best_epoch])

# Adam + LR Decay 추가
final_acc_decay = decay_acc[-1]
min_loss_decay = min(decay_loss)
best_epoch_decay = decay_acc.index(max(decay_acc)) + 1
summary_table.append(["Adam + LR Decay", final_acc_decay, min_loss_decay, best_epoch_decay])

# 표 출력
headers = ["Optimizer", "Final Test Accuracy", "Min Test Loss", "Best Epoch"]
print(tabulate(summary_table, headers=headers, floatfmt=".4f"))